FEATURE EXTRACTION

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import librosa
import math
from matplotlib.ticker import FormatStrFormatter

In [ ]:
# 1. Definizione del percorso del file
#CSV_INPUT_PATH = "audio_samples_metadata.csv"
CSV_OUTPUT_PATH = "audio_samples_filtered_features.csv"
INPUT_AUDIO_DIR = Path("AudioSamplesPreprocessed")
#SPECTROGRAM_DIR = Path("Spectrograms")
#SPECTROGRAM_DIR.mkdir(exist_ok=True)

CARICAMENTO DATI

In [ ]:
audio_files = list(INPUT_AUDIO_DIR.glob("*.wav"))

total_files = len(audio_files)

GENERAL PARAMETERS

In [ ]:
SAMPLE_RATE = 16000   #16 kHz
FRAME_SIZE = 2048
HOP_LENGTH = 512
BASE_SPLIT_FREQUENCY = 2000
N_MELS = 10

AMPLITUDE ENVELOPE

In [ ]:
def amplitude_envelope(signal, frame_size, hop_length):
    """Calculate the amplitude envelope of a signal with a given frame size and hop length."""
    amplitude_envelope = []
    
    # calculate amplitude envelope for each frame
    for i in range(0, len(signal), hop_length): 
        amplitude_envelope_current_frame = max(signal[i:i+frame_size])
        amplitude_envelope.append(amplitude_envelope_current_frame)
    
    return np.array(amplitude_envelope)

RMSE

In [ ]:
def rmse(signal, frame_size, hop_length):
    return librosa.feature.rms(y=signal, frame_length=frame_size, hop_length=hop_length)[0]

ZERO-CROSSING RATE

In [ ]:
def zero_crossing_rate(signal, frame_size, hop_length):
    return librosa.feature.zero_crossing_rate(y=signal, frame_length=frame_size, hop_length=hop_length)[0]

FOURIER TRANSFORM

In [ ]:
def magnitude_spectrum(signal):
    X = np.fft.fft(signal)
    X_mag = np.absolute(X)
    half = len(X_mag) // 2
    X_mag_half = X_mag[:half]
    return X_mag_half

SPECTOGRAM

In [ ]:
def spectrogram(signal, frame_size, hop_length):
    S_scale = librosa.stft(signal, n_fft=frame_size, hop_length=hop_length)
    #Y_scale = np.abs(S_scale) ** 2
    #Y_log_scale = librosa.power_to_db(Y_scale)
    #return Y_log_scale
    return np.abs(S_scale)

MEL SPECTROGRAM

In [ ]:
def mel_spectrogram(signal, frame_size, hop_length, sr, n_mels):
    mel_spectrogram = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

'''
filter_banks = librosa.filters.mel(n_fft=FRAME_SIZE, sr=sample_rate, n_mels=10)
'''

MFCC

In [ ]:
def mfcc(signal, sr, n_mfcc):
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
    return mfccs

def mfcc_delta(mfccs, order=1):
    delta_mfccs = librosa.feature.delta(mfccs, order=order)
    return delta_mfccs

BAND ENERGY RATIO

In [ ]:
def calculate_split_frequency_bin(split_frequency, sample_rate, num_frequency_bins):
    """Infer the frequency bin associated to a given split frequency."""
    
    frequency_range = sample_rate / 2
    frequency_delta_per_bin = frequency_range / num_frequency_bins
    split_frequency_bin = math.floor(split_frequency / frequency_delta_per_bin)
    return int(split_frequency_bin)

def band_energy_ratio(signal,split_frequency, frame_size, hop_length, sample_rate):
    """Calculate band energy ratio with a given split frequency."""

    epsilon = 1e-10

    spectrogram = librosa.stft(signal, n_fft=frame_size, hop_length=hop_length)

    split_frequency_bin = calculate_split_frequency_bin(split_frequency, sample_rate, spectrogram.shape[0])
    band_energy_ratio = []
    
    # calculate power spectrogram
    power_spectrogram = np.abs(spectrogram) ** 2
    power_spectrogram = power_spectrogram.T
    
    # calculate BER value for each frame
    for frame in power_spectrogram:
        sum_power_low_frequencies = frame[:split_frequency_bin].sum()
        sum_power_high_frequencies = frame[split_frequency_bin:].sum()
        band_energy_ratio_current_frame = sum_power_low_frequencies / (sum_power_high_frequencies + epsilon)
        band_energy_ratio.append(band_energy_ratio_current_frame)
    
    return np.array(band_energy_ratio)


SPECTRAL CENTROID

In [ ]:
def spectral_centroid(signal, sr, frame_size, hop_length):
    return librosa.feature.spectral_centroid(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]


SPECTRAL BANDWITH

In [ ]:
def spectral_bandwidth(signal, sr, frame_size, hop_length):
    return librosa.feature.spectral_bandwidth(y=signal, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]

FEATURE EXTRACTION

In [ ]:
def extract_stats(array):

    array = np.ravel(array)
    
    mean = np.mean(array)
    std = np.std(array)
    q1 = np.percentile(array, 25)
    median = np.percentile(array, 50)  # q2
    q3 = np.percentile(array, 75)
    iqr = q3 - q1
    delta_mean = np.mean(np.diff(array))
    min = np.min(array)
    max = np.max(array)
    
    return {
        'mean': mean,
        'std': std,
        'min': min,
        'max': max,
        'q1': q1,
        'median': median,
        'q3': q3,
        'iqr': iqr,
        'delta_mean':  delta_mean
    }

In [ ]:
all_audio_data = []

for i, filepath in enumerate(audio_files, 1):

    filename = filepath.name


    file_path = INPUT_AUDIO_DIR / filename

    
    audio_signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    #print(f"File caricato: {file_path}")
    
    ae = amplitude_envelope(audio_signal, FRAME_SIZE, HOP_LENGTH)
    rms = rmse(audio_signal, FRAME_SIZE, HOP_LENGTH)
    zcr = zero_crossing_rate(audio_signal, FRAME_SIZE, HOP_LENGTH)
    mel= mel_spectrogram(audio_signal, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE, n_mels=N_MELS)
    #spc = spectrogram(audio_signal, FRAME_SIZE, HOP_LENGTH)
    ber = band_energy_ratio(audio_signal, BASE_SPLIT_FREQUENCY, FRAME_SIZE, HOP_LENGTH, SAMPLE_RATE)
    sc = spectral_centroid(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)
    bw = spectral_bandwidth(audio_signal, SAMPLE_RATE, FRAME_SIZE, HOP_LENGTH)

    # --- Costruzione della riga per il DataFrame ---
    current_file_features = {'filename': filename}

    # 1. Processiamo le feature standard (1D)
    features_1d = {
        "ae": ae, "rms": rms, "zcr": zcr, 
        "ber": ber, "sc": sc, "bw": bw,
    }

    for name, signal in features_1d.items():
        stats = extract_stats(signal)
        for stat_name, value in stats.items():
            current_file_features[f"{name}_{stat_name}"] = value

    # 2. Processiamo il Mel Spectrogram (10 bande)
    # Calcoliamo le statistiche per OGNI banda di frequenza
    # mel ha shape (10, frames), quindi iteriamo sulle 10 righe
    for m in range(mel.shape[0]):
        band_stats = extract_stats(mel[m, :])
        for stat_name, value in band_stats.items():
            current_file_features[f"mel_b{m}_{stat_name}"] = value


    # Aggiungiamo la riga alla lista
    all_audio_data.append(current_file_features)
    print(f"[{i}/{len(audio_files)}] Feature estratte per: {filename}")

# --- Creazione del DataFrame finale ---
features_df = pd.DataFrame(all_audio_data)

In [ ]:
features_df.to_csv(CSV_OUTPUT_PATH, index=False)